In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
# Load data
df = pd.read_parquet('../dataset/fhvhv_tripdata_2025-05.parquet')

In [ ]:
# Filter days from 15 to 31
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df = df[df['pickup_datetime'].dt.day >= 25]

In [ ]:
df = 

,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag,cbd_congestion_fee
50,HV0003,B03404,B03404,2025-05-01 00:51:40,2025-05-01 00:54:02,2025-05-01 00:56:03,2025-05-01 01:20:42,36,217,4.68,...,0.00,0.0,0.0,15.54,Y,Y,N,N,N,0.0
310,HV0003,B03404,B03404,2025-05-01 00:50:45,2025-05-01 00:51:07,2025-05-01 00:52:03,2025-05-01 01:26:46,234,181,8.00,...,0.75,0.0,0.0,22.89,Y,Y,N,N,N,1.5
364,HV0003,B03404,B03404,2025-04-30 23:57:37,2025-05-01 00:03:55,2025-05-01 00:04:41,2025-05-01 00:12:12,186,113,1.58,...,0.75,0.0,0.0,3.39,Y,Y,N,N,N,1.5
369,HV0003,B03404,B03404,2025-04-30 23:59:03,2025-05-01 00:03:14,2025-05-01 00:04:22,2025-05-01 00:15:30,85,71,2.23,...,0.00,0.0,0.0,6.57,Y,Y,N,N,N,0.0
376,HV0003,B03404,B03404,2025-05-01 00:04:38,2025-05-01 00:11:34,2025-05-01 00:12:50,2025-05-01 00:43:26,37,188,5.58,...,0.00,0.0,1.0,15.25,Y,Y,N,N,N,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21091087,HV0003,B03404,B03404,2025-05-31 23:35:00,2025-05-31 23:40:25,2025-05-31 23:40:43,2025-06-01 00:19:31,32,191,18.74,...,0.00,0.0,0.0,44.85,Y,Y,N,N,N,0.0
21091101,HV0003,B03404,B03404,2025-05-31 23:49:01,2025-05-31 23:50:52,2025-05-31 23:51:37,2025-06-01 00:10:03,129,83,1.51,...,0.00,0.0,0.0,6.57,Y,Y,N,N,N,0.0
21091102,HV0003,B03404,B03404,2025-05-31 23:48:43,2025-05-31 23:52:05,2025-05-31 23:52:05,2025-06-01 00:08:39,129,83,1.51,...,0.00,0.0,0.0,0.00,Y,Y,N,N,N,0.0
21091177,HV0003,B03404,B03404,2025-05-31 23:38:58,2025-05-31 23:43:47,2025-05-31 23:45:18,2025-06-01 00:08:37,169,220,3.58,...,0.00,0.0,0.0,13.76,Y,Y,N,N,N,0.0


In [5]:
# Filter relevant columns
cols = ['shared_request_flag', 'PULocationID', 'DOLocationID',
        'trip_miles', 'trip_time', 'base_passenger_fare',
        'tolls', 'congestion_surcharge', 'tips',
        'request_datetime', 'pickup_datetime']

df = df[cols]

# Convert to datetime
df['request_datetime'] = pd.to_datetime(df['request_datetime'])
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# Wait time in minutes
df['wait_time'] = (df['pickup_datetime'] - df['request_datetime']).dt.total_seconds() / 60

# Hour of day
df['hour_of_day'] = df['pickup_datetime'].dt.hour

# Weekday vs Weekend
df['is_weekend'] = df['pickup_datetime'].dt.weekday.apply(lambda x: 1 if x >= 5 else 0)

# Drop original datetime columns
df = df.drop(['request_datetime', 'pickup_datetime'], axis=1)

# Clean missing values
df = df.dropna()

# Define X and y
X = df.drop('shared_request_flag', axis=1)
y = df['shared_request_flag'].map({'Y': 1, 'N': 0})

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define preprocessing and model pipeline
categorical = ['PULocationID', 'DOLocationID']
numerical = [col for col in X.columns if col not in categorical]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
    ])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

# Train model
model.fit(X_train, y_train)

# Evaluate
preds = model.predict(X_test)
print(classification_report(y_test, preds))

/var/folders/dh/n71j3h094h1bg5kjbx0gmdtw0000gn/T/ipykernel_14191/946022131.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['request_datetime'] = pd.to_datetime(df['request_datetime'])
/var/folders/dh/n71j3h094h1bg5kjbx0gmdtw0000gn/T/ipykernel_14191/946022131.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
/var/folders/dh/n71j3h094h1bg5kjbx0gmdtw0000gn/T/ipykernel_14191/946022131.py:14: SettingWithCopyWarning: 
A value is trying to be

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    267291

    accuracy                           1.00    267291
   macro avg       1.00      1.00      1.00    267291
weighted avg       1.00      1.00      1.00    267291



In [2]:
df = pd.read_parquet('../dataset/fhv_tripdata_2025-04.parquet')